# Build Production Training Data
Loads real production invoice and payment data, enriches with synthetic person/address/calendar data
generated by `generate_prod_dummy_data.ipynb`, and assembles the final flat training table.

**Inputs:**
- `data/raw/invoice_cohorts/person_invoice_history_2025-04-20.csv` — real production invoices (~24K rows)
- `data/raw/invoice_cohorts/payment_rows_2025-04-20.csv` — real production payment rows (~19K rows)
- `data/processed/prod_dummy_persons.csv` — synthetic persons (real IDs)
- `data/processed/prod_dummy_addresses.csv` — synthetic addresses (real census zip codes)
- `data/processed/prod_dummy_calendar_events.csv` — synthetic calendar events
- `data/processed/census_zip_features_model_ready.csv` — census zip features

**Output:** `data/processed/training_data_prod.csv`

> **Note:** `origin` (payment channel) is not on the `invoices` table — it lives on `payment_log`.
> This notebook derives `payment_origin_mode` per person (mode of origin across successful payments)
> and uses that as the feature instead of a per-invoice origin.

## Step 1: Load Raw Data
Loads production invoices and payments alongside the generated dummy data files.
Filters out invoices that have no linked `personid` (they cannot contribute to person-level features).

In [ ]:
import pandas as pd
import numpy as np

RAW_DIR = '../../data/raw/invoice_cohorts'
PROCESSED_DIR = '../../data/processed'

# --- Production data ---
invoices_df = pd.read_csv(f'{RAW_DIR}/person_invoice_history_2025-04-20.csv')
payment_rows_df = pd.read_csv(f'{RAW_DIR}/payment_rows_2025-04-20.csv')

# Filter out invoices without a personid
invoices_df = invoices_df[invoices_df['personid'].notna()].copy()

# Parse timestamps
# format='mixed' is required: some rows have sub-second precision (e.g. 2022-12-15 17:16:46.426343+00:00)
# and some do not (e.g. 2021-02-24 19:07:39+00:00). Without format='mixed', pandas locks on the first
# format it encounters and silently produces NaT for the 1,116 rows that don't match.
invoices_df['createdat'] = pd.to_datetime(invoices_df['createdat'], utc=True, errors='coerce', format='mixed')
payment_rows_df['submittedat'] = pd.to_datetime(
    payment_rows_df['submittedat'], utc=True, errors='coerce', format='mixed'
)

# --- Generated dummy data ---
persons_df = pd.read_csv(f'{PROCESSED_DIR}/prod_dummy_persons.csv')
addresses_df = pd.read_csv(f'{PROCESSED_DIR}/prod_dummy_addresses.csv', dtype={'postal_code': str})
calendar_events_df = pd.read_csv(f'{PROCESSED_DIR}/prod_dummy_calendar_events.csv')

# --- Census features ---
census_df = pd.read_csv(f'{PROCESSED_DIR}/census_zip_features_model_ready.csv', dtype={'zip_code': str})

print(f"Production invoices:    {len(invoices_df):,} rows | {invoices_df['personid'].nunique():,} unique persons")
print(f"Production payments:    {len(payment_rows_df):,} rows")
print(f"Dummy persons:          {len(persons_df)} rows")
print(f"Dummy addresses:        {len(addresses_df)} rows")
print(f"Dummy calendar events:  {len(calendar_events_df):,} rows")
print(f"Census zip codes:       {len(census_df):,} rows")

## Step 2: Derive Person-Level Features
Derives three groups of person-level features:

- **`appointment_reliability_score`** *(synthetic)*: ratio of COMPLETED+CONFIRMED appointments to total, from dummy calendar events
- **`average_days_to_pay`** *(real)*: mean days between invoice `createdat` and first successful payment `submittedat`, from production payment data; `-1` = no payment history
- **`payment_origin_mode`** *(real)*: most common payment channel (`origin`) across a person's successful payments; `0` = unknown/no history

> `origin` lives on `payment_log`, not on `invoices`, so it cannot be joined per-invoice.
> Per-person mode is the best available proxy.

In [ ]:
# --- Derive appointment_reliability_score (from dummy calendar events) ---
events_by_person = calendar_events_df.groupby('attendee_id').agg(
    total_appointments=('id', 'count'),
    completed_or_confirmed=('attendee_status', lambda x: x.isin(['COMPLETED', 'CONFIRMED']).sum()),
).reset_index()
events_by_person['appointment_reliability_score'] = (
    events_by_person['completed_or_confirmed'] / events_by_person['total_appointments']
).round(3)

# --- Derive average_days_to_pay (from real production payment data) ---
# First successful payment (paymentstatus=1) per invoice
first_payment_per_invoice = (
    payment_rows_df[payment_rows_df['paymentstatus'] == 1]
    .groupby('invoiceid')['submittedat']
    .min()
    .reset_index()
    .rename(columns={'submittedat': 'first_paid_at'})
)

# Join to invoices to get createdat and personid
invoice_pay_times = invoices_df[['id', 'personid', 'createdat']].merge(
    first_payment_per_invoice,
    left_on='id',
    right_on='invoiceid',
    how='inner'
)
invoice_pay_times['days_to_pay'] = (
    invoice_pay_times['first_paid_at'] - invoice_pay_times['createdat']
).dt.days

# Average per person
avg_days_by_person = invoice_pay_times.groupby('personid').agg(
    average_days_to_pay=('days_to_pay', 'mean'),
    total_paid_invoices=('days_to_pay', 'count'),
).reset_index()
avg_days_by_person['average_days_to_pay'] = avg_days_by_person['average_days_to_pay'].round(1)

# --- Derive payment_origin_mode (from real production payment data) ---
# origin is on payment_log; join successful payments back to invoices to get personid
payments_with_person = (
    payment_rows_df[payment_rows_df['paymentstatus'] == 1]
    .merge(
        invoices_df[['id', 'personid']],
        left_on='invoiceid',
        right_on='id',
        how='left',
        suffixes=('_payment', '_invoice')
    )
)
origin_mode_by_person = (
    payments_with_person[payments_with_person['personid'].notna()]
    .groupby('personid')['origin']
    .agg(lambda x: int(x.mode().iloc[0]) if len(x) > 0 else 0)
    .reset_index()
    .rename(columns={'origin': 'payment_origin_mode'})
)

# --- Combine all derived features ---
derived_features = persons_df[['id']].merge(
    events_by_person[['attendee_id', 'appointment_reliability_score', 'total_appointments']],
    left_on='id', right_on='attendee_id', how='left'
).merge(
    avg_days_by_person[['personid', 'average_days_to_pay', 'total_paid_invoices']],
    left_on='id', right_on='personid', how='left'
).merge(
    origin_mode_by_person[['personid', 'payment_origin_mode']],
    left_on='id', right_on='personid', how='left'
)

# Fill defaults
derived_features['average_days_to_pay'] = derived_features['average_days_to_pay'].fillna(-1)
derived_features['appointment_reliability_score'] = derived_features['appointment_reliability_score'].fillna(0.5)
derived_features['payment_origin_mode'] = derived_features['payment_origin_mode'].fillna(0).astype(int)

derived_features = derived_features[[
    'id', 'appointment_reliability_score', 'total_appointments',
    'average_days_to_pay', 'total_paid_invoices', 'payment_origin_mode'
]]

print(f"✅ Derived person-level features for {len(derived_features)} persons")
print(f"\nAppointment reliability score (synthetic):")
print(f"   Mean:   {derived_features['appointment_reliability_score'].mean():.3f}")
print(f"   Median: {derived_features['appointment_reliability_score'].median():.3f}")
has_history = derived_features[derived_features['average_days_to_pay'] > 0]
print(f"\nAverage days to pay (real production data):")
print(f"   Persons with payment history: {len(has_history)}/{len(derived_features)}")
if len(has_history) > 0:
    print(f"   Mean:   {has_history['average_days_to_pay'].mean():.1f} days")
    print(f"   Median: {has_history['average_days_to_pay'].median():.1f} days")
print(f"\nPayment origin mode distribution:")
print(origin_mode_by_person['payment_origin_mode'].value_counts().sort_index().to_string())
print(f"\nPreview:")
derived_features.head()

## Step 3: Enrich Persons with Census Data + Derived Features
Merges persons with their dummy addresses to get `postal_code`, then joins with census data
to attach income and demographic features. Also joins the derived behavioral features from Step 2.

In [ ]:
# Only use zip codes with complete, valid census data
valid_census = census_df[
    (census_df['census_match'] == 1) &
    (census_df['median_household_income'].notna()) &
    (census_df['poverty_rate_pct'].notna()) &
    (census_df['poverty_rate_pct'] >= 0) &
    (census_df['unemployment_rate_pct'] >= 0) &
    (census_df['average_household_size'].notna()) &
    (census_df['average_household_size'] > 0)
].copy()

print(f"Census zip codes total:     {len(census_df)}")
print(f"Valid zip codes (complete): {len(valid_census)}")

# Merge persons with addresses to get postal_code
persons_with_zip = persons_df[['id', 'is_guardian', 'entry_date']].merge(
    addresses_df[['person_id', 'postal_code']],
    left_on='id',
    right_on='person_id',
    how='left'
)

# Join with census features on zip code
persons_enriched = persons_with_zip.merge(
    valid_census[[
        'zip_code', 'median_household_income', 'mean_household_income',
        'poverty_rate_pct', 'unemployment_rate_pct',
        'private_insurance_coverage_pct', 'public_insurance_coverage_pct',
        'average_household_size', 'bachelors_or_higher_pct'
    ]],
    left_on='postal_code',
    right_on='zip_code',
    how='left'
)

# Join with derived behavioral + origin features
persons_enriched = persons_enriched.merge(
    derived_features[[
        'id', 'appointment_reliability_score', 'average_days_to_pay', 'payment_origin_mode'
    ]],
    on='id',
    how='left'
)

census_matched = persons_enriched['median_household_income'].notna().sum()
print(f"\n✅ Census + derived features join complete")
print(f"   Persons with census data: {census_matched}/{len(persons_enriched)} ({census_matched / len(persons_enriched) * 100:.1f}%)")
print(f"   Median household income range: ${persons_enriched['median_household_income'].min():,.0f} – ${persons_enriched['median_household_income'].max():,.0f}")
print(f"   Mean poverty rate: {persons_enriched['poverty_rate_pct'].mean():.1f}%")
print(f"   Appointment reliability (mean): {persons_enriched['appointment_reliability_score'].mean():.3f}")
print(f"   Avg days to pay (excl. no-history): {persons_enriched[persons_enriched['average_days_to_pay'] > 0]['average_days_to_pay'].mean():.1f}")
print(f"\nPreview:")
persons_enriched[['id', 'postal_code', 'median_household_income', 'appointment_reliability_score', 'average_days_to_pay']].head()

## Step 4: Assemble Flat Training Table
Joins production invoices with enriched person data and derives time-based and behavioral features.

**Feature groups:**
- **Invoice:** `amount`, `created_day_of_week`, `created_day_of_month`, `created_hour`, `payment_origin_mode`, `surchargingenabled`
- **Person:** `is_guardian`, `account_age_days`, `tenure_months`, `is_new_patient`
- **Derived behavioral (real):** `average_days_to_pay`
- **Derived behavioral (synthetic):** `appointment_reliability_score`
- **Census:** `median_household_income`, `poverty_rate_pct`, `average_household_size`, `bachelors_or_higher_pct`, `unemployment_rate_pct`

**Target:** `paid_within_30_days` = 1 if first successful payment was made within 30 calendar days of invoice creation, else 0 (includes unpaid and late-paid)

In [ ]:
# Merge production invoices with enriched person data
flat_df = invoices_df.merge(
    persons_enriched,
    left_on='personid',
    right_on='id',
    how='left',
    suffixes=('', '_person')
)

# --- Time-based features from invoice creation date ---
flat_df['created_day_of_week'] = flat_df['createdat'].dt.dayofweek
flat_df['created_day_of_month'] = flat_df['createdat'].dt.day   # "Payday" feature: bills near 1st/15th pay faster
flat_df['created_hour'] = flat_df['createdat'].dt.hour

# --- Account age features ---
flat_df['entry_date_ts'] = pd.to_datetime(flat_df['entry_date'], utc=True, errors='coerce')
flat_df['account_age_days'] = (flat_df['createdat'] - flat_df['entry_date_ts']).dt.days.clip(lower=0)
flat_df['tenure_months'] = flat_df['account_age_days'] // 30

# --- Cold start flag: no prior payment history (average_days_to_pay == -1) ---
flat_df['is_new_patient'] = (flat_df['average_days_to_pay'] == -1).astype(int)

# --- Target variable: paid_within_30_days ---
# 1 if the first successful payment was made within 30 calendar days of invoice creation
flat_df = flat_df.merge(
    invoice_pay_times[['id', 'days_to_pay']].rename(columns={'id': 'invoice_id_pay'}),
    left_on='id',
    right_on='invoice_id_pay',
    how='left'
)
flat_df['paid_within_30_days'] = (
    flat_df['days_to_pay'].notna() & (flat_df['days_to_pay'] <= 30)
).astype(int)

# --- Normalize bool-string columns to int ---
flat_df['surchargingenabled'] = flat_df['surchargingenabled'].map(
    {True: 1, False: 0, 'True': 1, 'False': 0}
).fillna(0).astype(int)
flat_df['is_guardian'] = flat_df['is_guardian'].map(
    {True: 1, False: 0, 'True': 1, 'False': 0}
).fillna(0).astype(int)

# --- Select final feature columns ---
feature_columns = [
    # Invoice features
    'amount',
    'created_day_of_week',
    'created_day_of_month',
    'created_hour',
    'payment_origin_mode',           # per-person mode; origin is on payment_log, not invoices
    'surchargingenabled',
    # Person features
    'is_guardian',
    'account_age_days',
    'tenure_months',
    'is_new_patient',
    # Derived behavioral features
    'average_days_to_pay',           # real production data
    'appointment_reliability_score', # synthetic (dummy calendar events)
    # Census features
    'median_household_income',
    'poverty_rate_pct',
    'average_household_size',
    'bachelors_or_higher_pct',
    'unemployment_rate_pct',
]

output_columns = ['personid', 'postal_code'] + feature_columns + ['status', 'paid_within_30_days']
training_df = flat_df[output_columns].copy()

print(f"✅ Flat training table assembled: {training_df.shape[0]:,} rows x {training_df.shape[1]} columns")
print(f"\nTarget distribution (paid_within_30_days):")
print(f"   Paid within 30d (1): {training_df['paid_within_30_days'].sum():,} ({training_df['paid_within_30_days'].mean() * 100:.1f}%)")
print(f"   Late or unpaid  (0): {(training_df['paid_within_30_days'] == 0).sum():,} ({(1 - training_df['paid_within_30_days'].mean()) * 100:.1f}%)")
print(f"\nNew features:")
print(f"   is_new_patient:        {training_df['is_new_patient'].sum():,} ({training_df['is_new_patient'].mean() * 100:.1f}%)")
print(f"   payment_origin_mode:   {sorted(training_df['payment_origin_mode'].unique().tolist())}")
print(f"   created_day_of_month:  {training_df['created_day_of_month'].min()}–{training_df['created_day_of_month'].max()}")
print(f"\nFeature columns ({len(feature_columns)}):")
for col in feature_columns:
    print(f"   {col}")
print(f"\nNull check:")
print(training_df[feature_columns].isnull().sum().to_string())
print(f"\nPreview:")
training_df.head()

## Step 5: Save Training Data
Saves the flat training table to `data/processed/training_data_prod.csv`.

In [ ]:
import os

output_dir = '../../data/processed'
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'training_data_prod.csv')
training_df.to_csv(output_path, index=False)

print(f"✅ Saved production training table: {output_path}")
print(f"   {len(training_df):,} rows x {len(training_df.columns)} columns")
print(f"\nSaved to: {os.path.abspath(output_path)}")
print(f"\nNext step: train and evaluate a model on training_data_prod.csv.")